# Les 09 - Metacognitief ontwerppatroon


## Installatie

Deze notebook demonstreert het Metacognition-ontwerppatroon met behulp van het Microsoft Agent Framework.

**Vereisten:**
- Azure OpenAI-implementatie geconfigureerd via omgevingsvariabelen
- Azure CLI geauthenticeerd (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Wat is Metacognitie?

Metacognitie is **nadenken over denken**. In de context van AI-agenten betekent het dat je agenten bouwt die kunnen:

- **Zelfreflectie toepassen** op hun eigen output en redeneerproces
- **Fouten detecteren** en zich gracieus herstellen in plaats van stilletjes te falen
- **Evalueren** of hun antwoorden volledig en nuttig zijn
- **Aanpassen** van hun strategie wanneer een eerste aanpak niet werkt (bijv. terugvallen op een back-upsysteem)

Een metacognitieve agent beantwoordt niet alleen vragen — hij houdt zijn eigen prestaties in de gaten en past zich ter plekke aan.


## Primaire en back-up tools

Een veelvoorkomend metacognitiepatroon is de **fallback-strategie**. De agent probeert eerst een primair hulpmiddel; als dit faalt (bijv. een 404-fout), herkent de agent de fout en schakelt transparant over naar een reservehulpmiddel.

Dit weerspiegelt systemen in de echte wereld waarbij primaire diensten mogelijk niet beschikbaar zijn en agenten het probleem zelf moeten diagnosticeren voordat ze een alternatieve route kiezen.

Hieronder definiëren we twee hulpmiddelen voor het opzoeken van vluchten:
- **Primaire** — dekt Parijs, Tokio en Barcelona
- **Back-up** — dekt Berlijn, Sydney en New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Zelfreflecterende agent met foutherstel

De onderstaande agent is geïnstrueerd om eerst het primaire vluchtsysteem te proberen, storingen te herkennen en op transparante wijze terug te vallen op het back-upsysteem. Na elk antwoord evalueert het kort of het de vraag van de gebruiker volledig heeft beantwoord.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Patroon voor zelfevaluatie

Een ander aspect van metacognitie is **zelfevaluatie**: een aparte agent (of dezelfde agent bij een tweede doorloop) beoordeelt een antwoord op volledigheid, nauwkeurigheid en behulpzaamheid.

Hieronder maken we een `ResponseEvaluator` agent die reacties van de reisagent op drie dimensies beoordeelt.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Samenvatting

In deze les heb je geleerd hoe je **metacognitieve agenten** bouwt met behulp van het Microsoft Agent Framework:

- **Zelfreflectie**: Agenten die hun eigen redenering monitoren en op transparante wijze communiceren wat er gebeurde.
- **Foutherstel met fallbacks**: Een patroon met primaire + back-up tool waarbij de agent fouten detecteert (bijv. 404-fouten) en automatisch een alternatieve bron probeert.
- **Zelfevaluatie**: Een aparte evaluator-agent die antwoorden beoordeelt op volledigheid, nauwkeurigheid en behulpzaamheid.

Deze patronen maken agenten robuuster, transparanter en betrouwbaarder — essentiële eigenschappen voor productie-implementaties.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Disclaimer:
Dit document is vertaald met behulp van de AI-vertalingsdienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet als de gezaghebbende bron worden beschouwd. Voor cruciale informatie wordt een professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
